# Trashball Exercise

## 1) Title & Objective
- Challengebeschrijving
- Doelvariabele en succescriteria
- Vergelijking: **Logistic Regression vs KNN**

In [ ]:
# 2) Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

In [ ]:
# 3) Data Loading & First Look
DATA_PATH = "/content/your_dataset.csv"
df = pd.read_csv(DATA_PATH)

display(df.head())
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())

## 4) EDA (Exploratory Data Analysis)
- Target class balance
- Univariate analyse
- Correlatie-overzicht
- Korte observaties

In [ ]:
# Vervang TARGET_COL met je echte targetnaam
TARGET_COL = "target"

df[TARGET_COL].value_counts(dropna=False).plot(kind="bar", title="Target balance")
plt.show()

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
if TARGET_COL in num_cols:
    num_cols.remove(TARGET_COL)

df[num_cols].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(df[num_cols + [TARGET_COL]].corr(numeric_only=True), annot=False, cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

In [ ]:
# 5) Data Preparation
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# 6) Baseline Model: Logistic Regression
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1]

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print("LR F1:", f1_score(y_test, y_pred_lr))

In [ ]:
# 7) Logistic Regression - Feature Importance
coef = lr_pipe.named_steps["model"].coef_.ravel()
importance = pd.DataFrame({
    "feature": X.columns,
    "coef": coef,
    "abs_coef": np.abs(coef)
}).sort_values("abs_coef", ascending=False)

display(importance.head(10))

sns.barplot(data=importance.head(10), x="abs_coef", y="feature")
plt.title("Top 10 LR feature importances (|coef|)")
plt.show()

In [ ]:
# 8) Evaluation Metrics (Logistic Regression)
cm_lr = confusion_matrix(y_test, y_pred_lr)
print("Confusion Matrix (LR):\n", cm_lr)
print("\nClassification Report (LR):\n")
print(classification_report(y_test, y_pred_lr))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

plt.plot(fpr_lr, tpr_lr, label=f"LR AUC = {auc_lr:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()

In [ ]:
# 9) KNN Model + eenvoudige k-tuning
k_values = range(1, 26)
scores = []

for k in k_values:
    knn_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])
    knn_pipe.fit(X_train, y_train)
    y_pred_k = knn_pipe.predict(X_test)
    scores.append(f1_score(y_test, y_pred_k))

best_k = int(k_values[np.argmax(scores)])
print("Best k based on F1:", best_k)

knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=best_k))
])
knn_pipe.fit(X_train, y_train)
y_pred_knn = knn_pipe.predict(X_test)
y_proba_knn = knn_pipe.predict_proba(X_test)[:, 1]

In [ ]:
# 10) Evaluation Metrics (KNN)
cm_knn = confusion_matrix(y_test, y_pred_knn)
print("Confusion Matrix (KNN):\n", cm_knn)
print("\nClassification Report (KNN):\n")
print(classification_report(y_test, y_pred_knn))

fpr_knn, tpr_knn, _ = roc_curve(y_test, y_proba_knn)
auc_knn = roc_auc_score(y_test, y_proba_knn)

plt.plot(fpr_knn, tpr_knn, label=f"KNN AUC = {auc_knn:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - KNN")
plt.legend()
plt.show()

In [ ]:
# 11) Model Comparison & Final Choice
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_lr),
        "Precision": precision_score(y_test, y_pred_lr),
        "Recall": recall_score(y_test, y_pred_lr),
        "F1": f1_score(y_test, y_pred_lr),
        "AUC": auc_lr
    },
    {
        "Model": "KNN",
        "Accuracy": accuracy_score(y_test, y_pred_knn),
        "Precision": precision_score(y_test, y_pred_knn),
        "Recall": recall_score(y_test, y_pred_knn),
        "F1": f1_score(y_test, y_pred_knn),
        "AUC": auc_knn
    }
])

display(results.sort_values("F1", ascending=False))

## 12) Conclusion & Next Steps
- Schrijf hier je korte conclusie (3-5 bullets).
- Kies het finale model met motivatie.
- Mogelijke verbeteringen: feature engineering, hyperparameter tuning, cross-validation, threshold tuning.